# Build a churn classifier and evaluate it with the right metric

One row is one telco customer; `Churn` is `Yes` when the customer left in the last month. The retention manager: "My team calls the customers you flag and offers a discount. A call costs us little. A customer who leaves costs us a year of bills."

## The starter

The cell repeats D01 5.6: `TotalCharges` converted to numbers, the `ColumnTransformer` `pre`, and the 80 / 20 split with `stratify=y, random_state=42`, so your 1,409 test customers are the same as in the 4.2 demonstration.

In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv("data/telco_churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
y = (df["Churn"] == "Yes").astype(int)
X = df.drop(columns=["customerID", "Churn"])
num_cols = list(X.select_dtypes(include="number").columns)
cat_cols = list(X.select_dtypes(include=["str", "object"]).columns)
pre = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
print(len(X_train), "training customers,", len(X_test), "test customers,", y_test.sum(), "test churners")

5634 training customers, 1409 test customers, 374 test churners


## Step 1 · Fit and score against a dummy

Fit `make_pipeline(pre, LogisticRegression(max_iter=1000))`. Print the test accuracy and the accuracy of `DummyClassifier(strategy="most_frequent")`.

In [2]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

model = make_pipeline(pre, LogisticRegression(max_iter=1000)).fit(X_train, y_train)
pred = model.predict(X_test)
dummy_acc = accuracy_score(y_test, DummyClassifier(strategy="most_frequent").fit(X_train, y_train).predict(X_test))
print(f"model {accuracy_score(y_test, pred):.3f} | dummy {dummy_acc:.3f}")

model 0.806 | dummy 0.735


## Step 2 · The confusion matrix

Print `confusion_matrix(y_test, model.predict(X_test))`. Write down what each of the four cells counts, in words: "customers who stayed and whom we did not flag", and so on.

In [3]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, pred)
print(cm)
tn, fp, fn, tp = cm.ravel()
print(f"TN {tn}: stayed, not flagged | FP {fp}: stayed, flagged (a wasted call)")
print(f"FN {fn}: churned, not flagged (a missed churner) | TP {tp}: churned, flagged")

[[926 109]
 [165 209]]
TN 926: stayed, not flagged | FP 109: stayed, flagged (a wasted call)
FN 165: churned, not flagged (a missed churner) | TP 209: churned, flagged


## Step 3 · Precision, recall and F1

Print `classification_report(y_test, pred, digits=3)`. Find precision, recall and F1 for the churn class (`1`), and check precision and recall by hand from your four cells.

In [4]:
from sklearn.metrics import classification_report

print(classification_report(y_test, pred, digits=3))
print(f"by hand: precision {tp / (tp + fp):.3f}, recall {tp / (tp + fn):.3f}")

              precision    recall  f1-score   support

           0      0.849     0.895     0.871      1035
           1      0.657     0.559     0.604       374

    accuracy                          0.806      1409
   macro avg      0.753     0.727     0.738      1409
weighted avg      0.798     0.806     0.800      1409

by hand: precision 0.657, recall 0.559


## Step 4 · Ranking metrics

Compute `roc_auc_score` and `average_precision_score` from `model.predict_proba(X_test)[:, 1]`. Compare the average precision with the share of churners in the test rows.

In [5]:
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score, roc_auc_score

proba = model.predict_proba(X_test)[:, 1]
print(f"ROC AUC {roc_auc_score(y_test, proba):.3f} | average precision {average_precision_score(y_test, proba):.3f} "
      f"| base rate {y_test.mean():.3f}")

ROC AUC 0.842 | average precision 0.633 | base rate 0.265


## Step 5 · A second model with `class_weight="balanced"`

Fit `LogisticRegression(max_iter=1000, class_weight="balanced")` in the same pipeline. Repeat steps 2 to 4 for it.

In [6]:
from sklearn.base import clone

bal = make_pipeline(clone(pre), LogisticRegression(max_iter=1000, class_weight="balanced")).fit(X_train, y_train)
pred_b, proba_b = bal.predict(X_test), bal.predict_proba(X_test)[:, 1]
tn_b, fp_b, fn_b, tp_b = confusion_matrix(y_test, pred_b).ravel()
print(confusion_matrix(y_test, pred_b))
print(f"precision {precision_score(y_test, pred_b):.3f}, recall {recall_score(y_test, pred_b):.3f}, "
      f"F1 {f1_score(y_test, pred_b):.3f}, ROC AUC {roc_auc_score(y_test, proba_b):.3f}")

[[747 288]
 [ 81 293]]
precision 0.504, recall 0.783, F1 0.614, ROC AUC 0.841


## Step 6 · "Which model should my team use?"

Translate: which of her two costs is larger, a wasted call (a false positive) or a missed churner (a false negative)? Which metric counts the misses? Answer in two sentences with the numbers from both models.

In [7]:
print(f"A missed churner costs a year of bills and a call costs little, so recall matters most: the balanced "
      f"model finds {tp_b} of the {y_test.sum()} churners against {tp} ({tp_b - tp} more). The price is "
      f"{fp_b - fp} more calls to customers who would have stayed, which the manager says is cheap.")

A missed churner costs a year of bills and a call costs little, so recall matters most: the balanced model finds 293 of the 374 churners against 209 (84 more). The price is 179 more calls to customers who would have stayed, which the manager says is cheap.


## Check yourself

In [8]:
check = {
    "Step 1 · model accuracy": round(accuracy_score(y_test, pred), 3),
    "Step 1 · dummy accuracy": round(dummy_acc, 3),
    "Step 2 · TN, FP, FN, TP": f"{tn}, {fp}, {fn}, {tp}",
    "Step 3 · precision": round(precision_score(y_test, pred), 3),
    "Step 3 · recall": round(recall_score(y_test, pred), 3),
    "Step 3 · F1": round(f1_score(y_test, pred), 3),
    "Step 4 · ROC AUC": round(roc_auc_score(y_test, proba), 3),
    "Step 4 · average precision": round(average_precision_score(y_test, proba), 3),
    "Step 4 · base rate": round(y_test.mean(), 3),
    "Step 5 · balanced TN, FP, FN, TP": f"{tn_b}, {fp_b}, {fn_b}, {tp_b}",
    "Step 5 · balanced precision": round(precision_score(y_test, pred_b), 3),
    "Step 5 · balanced recall": round(recall_score(y_test, pred_b), 3),
    "Step 5 · balanced F1": round(f1_score(y_test, pred_b), 3),
    "Step 5 · balanced ROC AUC": round(roc_auc_score(y_test, proba_b), 3),
    "Step 6 · churners in the test rows": y_test.sum(),
    "Step 6 · extra churners found": tp_b - tp,
    "Step 6 · extra calls to customers who stayed": fp_b - fp,
}
for label, value in check.items():
    print(f"{label:<46} {value}")

Step 1 · model accuracy                        0.806
Step 1 · dummy accuracy                        0.735
Step 2 · TN, FP, FN, TP                        926, 109, 165, 209
Step 3 · precision                             0.657
Step 3 · recall                                0.559
Step 3 · F1                                    0.604
Step 4 · ROC AUC                               0.842
Step 4 · average precision                     0.633
Step 4 · base rate                             0.265
Step 5 · balanced TN, FP, FN, TP               747, 288, 81, 293
Step 5 · balanced precision                    0.504
Step 5 · balanced recall                       0.783
Step 5 · balanced F1                           0.614
Step 5 · balanced ROC AUC                      0.841
Step 6 · churners in the test rows             374
Step 6 · extra churners found                  84
Step 6 · extra calls to customers who stayed   179
